# Module 8 - Train on Tiny Shakespeare

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vritti-Dev/foundation-model/blob/main/notebooks/m08_train.ipynb)

## What you'll build

Now you train the GPT for real, on the **Tiny Shakespeare** corpus, and
watch it go from random noise to recognisable (if nonsensical) English.
This is the same training loop used everywhere, just scaled down: sample
random windows of text, predict the next character at every position,
backprop the cross-entropy loss, and step AdamW.

> **Run this on Colab** with a GPU for the best experience (it works on
> CPU too, just slower). Click the *Open in Colab* badge. A free
> **Google account** is required. Grading is by **pasting the printed
> token** into the browser grader.

The `train` helper in `reference/gpt/train.py` handles everything:
building the tokenizer, resizing the config's vocab, the AdamW loop, and
reporting initial vs final loss. You pick the corpus and the number of
iterations; more iterations means lower loss (down to a point).

In [ ]:
# Bootstrap: make the course code importable.
# On Colab this clones the repo and cds into it; locally it is a no-op.
import os, sys
try:
    import google.colab  # noqa: F401
    if not os.path.exists('foundation-model'):
        !git clone -q https://github.com/Vritti-Dev/foundation-model.git
    %cd foundation-model
    print('repo ready at', os.getcwd())
except ImportError:
    pass  # running locally; assume cwd is the repo root

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| GPU:', torch.cuda.is_available())


In [ ]:
# Get the Tiny Shakespeare text. On Colab you can download the full file;
# here we fall back to a small embedded sample so the notebook always runs.
SAMPLE = (
    'First Citizen:\nBefore we proceed any further, hear me speak.\n\n'
    'All:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather '
    'to die than to famish?\n\nAll:\nResolved. resolved.\n\n'
    'First Citizen:\nFirst, you know Caius Marcius is chief enemy to the '
    'people.\n\nAll:\nWe know\'t, we know\'t.\n'
)

try:
    import urllib.request
    url = ('https://raw.githubusercontent.com/karpathy/char-rnn/master/'
           'data/tinyshakespeare/input.txt')
    data = urllib.request.urlopen(url, timeout=10).read().decode('utf-8')
    print('downloaded tiny shakespeare:', len(data), 'chars')
except Exception as e:
    data = SAMPLE * 20
    print('download failed (', e, '); using embedded sample:', len(data), 'chars')


In [ ]:
# Train. On a GPU bump max_iters up (e.g. 2000+); on CPU keep it small.
from reference.gpt.train import train, make_submission_token
import torch

max_iters = 1000 if torch.cuda.is_available() else 100
config = 't4' if torch.cuda.is_available() else 'cpu'
res = train(config=config, max_iters=max_iters, seed=1337, data=data)
print(f"initial loss {res['initial_loss']:.4f} -> final loss {res['final_loss']:.4f}")


## Print your submission token

Copy the printed token line and paste it into the browser grader for
Module 8.

In [ ]:
print(make_submission_token(res, 'M8'))
